In [1]:
from ukrdc.database import Connection
from sqlalchemy.orm import sessionmaker
import datetime as dt
import plotly.graph_objects as go

to_time = dt.datetime.now()
from_time = to_time - dt.timedelta(days=90)

engine = Connection.get_engine_from_file(key="ukrdc_staging")

ukrdc3_sessionmaker = sessionmaker(
    autocommit=False, autoflush=False, bind=engine
)

ukrdc3 = ukrdc3_sessionmaker()

In [2]:
from ukrdc_stats.calculators.demographics_prd import RenalDiagnosisStatsCalculator

import plotly.graph_objects as go 
import plotly.express as px

import pandas as pd


calculator = RenalDiagnosisStatsCalculator(ukrdc3, "RNJ00")
calculator.extract_patient_cohort()


# run function to extract stats
output = calculator.extract_stats()
back_to_back = pd.DataFrame(output.gender.data.dict())

# Plot gender stats back to back 
data = [
    go.Bar(
        x = -back_to_back[back_to_back.x == "Male"].z,
        y = back_to_back[back_to_back.x == "Male"].y,
        orientation="h",
        name = "Male"
    ), 
    go.Bar(
        x = back_to_back[back_to_back.x == "Female"].z,
        y = back_to_back[back_to_back.x == "Female"].y,
        orientation="h",
        name = "Female"
    ) 
]

fig = go.Figure(
    data=data,
    layout = {
        "title" :{
            #"text": output.gender.metadata.title,
            "text": "PRD ",
            "x" : 0.5,
            "xanchor" : "center" 
        }
    }
)
fig.show()

# plot age
stacked_bar_data = pd.DataFrame(output.age.data.dict())
stacked_bar_data.x = pd.to_numeric(stacked_bar_data.x)
stacked_bar_data.rename(columns={"x":"Age", "y":"Primary Renal Diagnosis", "z":"Patients"}, inplace = True)

fig = px.bar(stacked_bar_data, x="Age", y = "Patients", color = "Primary Renal Diagnosis")
fig.show()



# plot ethnicity
ethnicity_data = pd.DataFrame(output.ethnic_group.data.dict())
ethnicity_data.rename(columns={"x":"Ethnic Group", "y":"Primary Renal Diagnosis", "z":"Patients"}, inplace = True)
fig = px.sunburst(
    ethnicity_data, 
    path = ["Ethnic Group", "Primary Renal Diagnosis"],
    values = "Patients"
)
fig.show()